In [1]:
# Enable auto-reload so that changes to imported modules are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

In [2]:
from ipywidgets import FloatProgress, Layout
from IPython.display import display
import micasense.imageset as imageset
import micasense.capture as capture
import os, glob
import multiprocessing

panelNames = None
useDLS = True  # Use Downwelling Light Sensor if no calibration panel is available

# ----- Input paths -----
imagePath = '../../../data/udine/original_data/captures'
panelPath = '../../../data/udine/original_data/panel'
panelNames = glob.glob(os.path.join(panelPath,'IMG_0000_*.tif'))  # Calibration panel images

# ----- Output paths -----
outputPath = '../../../data/udine/preprocessing/stacks'       # Aligned multispectral stacks
thumbnailPath = '../../../data/udine/preprocessing/rgb'        # RGB thumbnail previews

overwrite = False          # Set to False to resume interrupted processing
generateThumbnails = True  # Generate RGB preview for each capture

# Load panel captures and extract reflectance calibration data.
# If panel images are not provided, radiance images will be used instead.
# Note: for firmware before Altum 1.3.5 / RedEdge 5.1.7, panel_reflectance_by_band
# must be set manually.
if panelNames is not None:
    panelCap = capture.Capture.from_filelist(panelNames)
    print('Panel captures loaded successfully.')
else:
    panelCap = None
    print('Panel captures not loaded.')

# Determine panel reflectance and compute irradiance for reflectance conversion
if panelCap is not None:
    if panelCap.panel_albedo() is not None:
        panel_reflectance_by_band = panelCap.panel_albedo()
        print('Panel reflectance loaded successfully.')
    else:
        panel_reflectance_by_band = [0.65]*len(panelCap.images)  # Approximate fallback value
        print('Panel reflectance not loaded. Using approximate reflectance.')
    panel_irradiance = panelCap.panel_irradiance(panel_reflectance_by_band)
    img_type = "reflectance"
else:
    # Fallback: use DLS-based reflectance or raw radiance
    if useDLS:
        img_type='reflectance'
        print('Using reflectance DLS.')
    else:
        img_type = "radiance"
        print('Using radiance.')


Panel captures loaded successfully.
Panel reflectance loaded successfully.


In [3]:
# Display a progress bar while loading the full image set from disk
f = FloatProgress(min=0, max=1, layout=Layout(width='100%'), description="Loading")
display(f)
def update_f(val):
    if (val - f.value) > 0.005 or val == 1:  # Throttle updates to reduce CPU overhead
        f.value=val

# Scan the image directory and group individual band files into multi-band captures
%time imgset = imageset.ImageSet.from_directory(imagePath, progress_callback=update_f)
update_f(1.0)


FloatProgress(value=0.0, description='Loading', layout=Layout(width='100%'), max=1.0)

CPU times: user 503 ms, sys: 40.5 ms, total: 544 ms
Wall time: 3.34 s


In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import micasense.imageutils as imageutils
import micasense.plotutils as plotutils

# Load a single capture to compute the alignment (warp) matrices
alignment_img = glob.glob(os.path.join(imagePath,'IMG_0036_*.tif'))
cpt = capture.Capture.from_filelist(alignment_img)

# ----- Alignment settings -----
match_index = 4  # Reference band index for alignment
                  # 0: Blue, 1: Green, 2: Red, 3: NIR, 4: Red Edge
                  # 5: Blue-444, 6: Green-531, 7: Red-650
                  # 8: Red edge-705, 9: Red edge-740

max_alignment_iterations = 20  # Higher values may improve accuracy but increase runtime
warp_mode = cv2.MOTION_HOMOGRAPHY  # Use HOMOGRAPHY for Altum; AFFINE is an alternative
pyramid_levels = 3  # Multi-resolution pyramid depth for 10-band imagery

print("Alinging images. Depending on settings this can take from a few seconds to many minutes")

# Compute warp matrices that align all bands to the reference band
warp_matrices, alignment_pairs = imageutils.align_capture(cpt,
                                                          ref_index = match_index,
                                                          max_iterations = max_alignment_iterations,
                                                          warp_mode = warp_mode,
                                                          pyramid_levels = pyramid_levels)

print("Finished Aligning, warp matrices={}".format(warp_matrices))


Alinging images. Depending on settings this can take from a few seconds to many minutes
Finished aligning band 4
Finished aligning band 0
Finished aligning band 8
Finished aligning band 6
Finished aligning band 7
Finished aligning band 3
Finished aligning band 9
Finished aligning band 5
Finished aligning band 1
Finished aligning band 2
Finished Aligning, warp matrices=[array([[ 1.0040864e+00, -7.4430788e-03, -3.3286095e+01],
       [ 7.0750872e-03,  1.0027938e+00, -7.7721471e-01],
       [ 1.1506276e-06, -1.8202343e-06,  1.0000000e+00]], dtype=float32), array([[ 1.0003500e+00, -2.1300591e-03, -2.2057552e+01],
       [ 1.0440277e-03,  1.0018458e+00, -5.8781284e-01],
       [-2.2985755e-06, -2.5595128e-07,  1.0000000e+00]], dtype=float32), array([[ 9.9591613e-01, -5.0186333e-03, -1.6835062e+01],
       [ 3.8949682e-03,  9.9798226e-01,  1.2554541e+01],
       [-2.5367942e-06,  1.2353189e-07,  1.0000000e+00]], dtype=float32), array([[ 1.0020077e+00, -8.0345459e-03, -1.1813504e+01],
       

In [5]:
import exiftool
import datetime

use_multi_process = True    # Enable parallel processing for faster saving
overwrite_existing = False  # Skip already exported files

# Display a progress bar for the saving process
f2 = FloatProgress(min=0, max=1, layout=Layout(width='100%'), description="Saving")
display(f2)
def update_f2(val):
    f2.value=val

# Create output directories if they do not exist
if not os.path.exists(outputPath):
    os.makedirs(outputPath)
if generateThumbnails and not os.path.exists(thumbnailPath):
    os.makedirs(thumbnailPath)

# If no panel was provided, irradiance is None and DLS data will be used instead
try:
    irradiance = panel_irradiance+[0]
except NameError:
    irradiance = None

start_time = datetime.datetime.now()

# Apply warp matrices to all captures and save as aligned multi-band stacks (+ RGB thumbnails)
imgset.save_stacks(warp_matrices,
                     outputPath,
                     thumbnailPath,
                     irradiance = irradiance,
                     multiprocess=use_multi_process, 
                     overwrite=overwrite_existing, 
                     progress_callback=update_f2)

end_time = datetime.datetime.now()
update_f2(1.0)

# Print execution summary
print("Saving time: {}".format(end_time-start_time))
print("Alignment+Saving rate: {:.2f} captures per second".format(float(len(imgset.captures))/float((end_time-start_time).total_seconds())))



FloatProgress(value=0.0, description='Saving', layout=Layout(width='100%'), max=1.0)

Saving time: 0:00:44.406126
Alignment+Saving rate: 0.77 captures per second


In [6]:
# Rename output files from UUID-based names to the original capture ID (e.g. IMG_0281)
for capture in imgset.captures:
    filename = capture.images[0].meta.get_item("File:FileName")[:8]  # Extract capture ID

    for directory, suffix in zip((outputPath, thumbnailPath), ('.tif', '_rgb.jpg')):
        old_path = os.path.join(directory, f'{capture.uuid}{suffix}')
        new_path = os.path.join(directory, f'{filename}{suffix}')

        if os.path.exists(old_path):
            os.rename(old_path, new_path)
            print(f'Renamed: {os.path.basename(old_path)} -> {os.path.basename(new_path)}')


Renamed: pTsmMYC0FSNS57yBt5Qu.tif -> IMG_0036.tif
Renamed: pTsmMYC0FSNS57yBt5Qu_rgb.jpg -> IMG_0036_rgb.jpg
Renamed: 4z8HD5P3WVk11rUddSMp.tif -> IMG_0047.tif
Renamed: 4z8HD5P3WVk11rUddSMp_rgb.jpg -> IMG_0047_rgb.jpg
Renamed: 312u14rkq6yFDJKhdL3l.tif -> IMG_0070.tif
Renamed: 312u14rkq6yFDJKhdL3l_rgb.jpg -> IMG_0070_rgb.jpg
Renamed: BSfsqASpWRBq5lnJJ2pi.tif -> IMG_0081.tif
Renamed: BSfsqASpWRBq5lnJJ2pi_rgb.jpg -> IMG_0081_rgb.jpg
Renamed: LdACEnyQJ4DXMt2aH6Ku.tif -> IMG_0092.tif
Renamed: LdACEnyQJ4DXMt2aH6Ku_rgb.jpg -> IMG_0092_rgb.jpg
Renamed: uPTta8dHIJnSDe28VwCr.tif -> IMG_0102.tif
Renamed: uPTta8dHIJnSDe28VwCr_rgb.jpg -> IMG_0102_rgb.jpg
Renamed: tZ2aonSRDFzsSGKSwLcf.tif -> IMG_0117.tif
Renamed: tZ2aonSRDFzsSGKSwLcf_rgb.jpg -> IMG_0117_rgb.jpg
Renamed: IcBwF9SwnJGyUAMNoVtR.tif -> IMG_0163.tif
Renamed: IcBwF9SwnJGyUAMNoVtR_rgb.jpg -> IMG_0163_rgb.jpg
Renamed: oLmxTzQ1OST2yc94TdoD.tif -> IMG_0173.tif
Renamed: oLmxTzQ1OST2yc94TdoD_rgb.jpg -> IMG_0173_rgb.jpg
Renamed: Y4j7bxkJQW3RWkDHSWR